In [2]:
import csv
from pathlib import Path
import requests
from bs4 import BeautifulSoup
import numpy as np
from sortedcontainers import SortedDict
import time
import json

In [ ]:
# the following is not needed to run this file 

# however, it is a dependency needed to actually run the application so if you don't have the package installed yet you need to 

import streamlit

In [4]:
#**********************************************************************
# Purpose: request a user's manga list and return needed data to calculate recomendations and run user interface
#
# Precondition: passed valid username
#
# Postcondition: returns user rating information of rated manga and returns data of manga used in user interface
#
#***********************************************************************
def user_rated_list(username : str):

    all_entries = []
    offset = 0
    nodata = False
    
    while not nodata:
        url = f"https://myanimelist.net/mangalist/{username}/load.json?status=7&offset={offset}"

        r = requests.get(url)
        if r.ok:
            r.raise_for_status()

            data = r.json()

            if data:
                all_entries.extend(data)
                offset += len(data)
            else:
                nodata = True
        else: #if request satus is bad like private profile return empty
            return [],[]
        rows = []
        data = []

        for item in all_entries:
            if not item.get("score") == 0:
                
                if item["manga_english"] != "":
                    rows.append({
                        "title": item["manga_english"],
                        "score": item["score"]
                    })
                else:
                    rows.append({
                        "title": item["manga_title"],
                        "score": item["score"]
                    })

                tag_list = [] 
                for genre in item["genres"]:
                    tag_list.append({
                        
                        "id": genre["id"],
                        "name": genre["name"]
                    })
                image = item["manga_image_path"]
                link = item["manga_url"]
                
                jp_title = item["manga_title"]

                data.append((tag_list, image, link, jp_title))

        
        if not rows:# checks if empty 
            return ([],[])

        return rows, data


#**********************************************************************
# Purpose: retreive user and manga data 
#
# Precondition: is passed a valid argument list of users. if there is a file named "NameData.json" in the same directory it must be built by this program with the same username_list
#               that has been passed in to this function.    if there is a old json file and new username_list the file must be deleted
#               
# Postcondition: creates a file named "NameData.json" in the current directory as a save state and will shorten the process of this function if it is valid
#
#***********************************************************************
def user_data_list(userame_list):
    user_data = []
    file_path = Path("NameData.json")

    if file_path.is_file():
        with open("NameData.json", "r") as f:
            user_data = json.load(f)
    
    start = len(user_data)

    for ii, name in enumerate(userame_list):
        if ii <= start:
            continue
        else:
            try:
                user_data.append(user_rated_list(name))

                if ii % 100 == 1:
                    with open("NameData.json", "w") as file:
                        json.dump(user_data, file)

                time.sleep(1.2)
            except Exception as e:
                print(e)
                time.sleep(3)
    return user_data


#**********************************************************************
# Purpose: process users and manga then format their storage
#
# Precondition: has a list of of my anime list usernames to search, if there is a file named "NameData.json" in the same directory it must be built by this program with the same username_list
#               that has been passed in to this function. 
# 
#               If there is no "NameData.json" the function will create one to store user processing because large amount of requests can take days and can easily be interupted. 
#               "NameData.json" effectivly saves processing through a username list, however it is not built to compensate for using a new username list, you would need to delete the current
#               file if you wish to use a different amount of users in your processing
#               
#
# Postcondition: returns a dict of manga with a list of users with their ratings, user z-scores, and show z-scores. Returns a list of users with dicts of manga with ratings and user z-scores. 
#                additionaly returns extra manga data used in presentation and calculation. Returns a dictionary of individual manga genre tags and returns a dictionary genre_id used to translate genre tag numbers to words.
#                Finaly returns the global std and average to be used in later processing
#
#***********************************************************************
def make_collection(userame_list):

    manga_dict = SortedDict()
    user_list = [] # a list is used to make the user identification numbers line up with index. This wastes memory, there are better ways to do this.
    usercount = 0
    manga_genres = SortedDict()
    genre_ids = {}
    added_data = {}    

    user_data = user_data_list(userame_list)

    for user in user_data:
        list = user[0]
        data = user[1]

        try:
            if list:

                user_list.append(SortedDict())

                for i, dict in enumerate(list):#loops through dataframe
                    title = str(dict["title"])
                    score = dict["score"]
                    
                    row = [usercount, score]
                    user_list[usercount][title] = []


                    if title in manga_dict:
                        manga_dict[title].append(row)
                    else:
                        manga_genres[title] = []
                        for tag_dict in data[i][0]:

                            if tag_dict["id"] not in genre_ids:
                                genre_ids[tag_dict["id"]] = tag_dict["name"]# adds id translation to genre_ids

                            manga_genres[title].append(tag_dict["id"]) # adds genre id to manga for filtering later

                        added_data[title] = [data[i][1], data[i][2], data[i][3]]

                        manga_dict[title] = [row]

                    user_row = row[1:]
                    user_list[usercount][title] = user_row
                usercount +=1

        except Exception as e:
            print(e)
            
    global_std, global_average = global_normalization(user_list)
    

    users_rating_normalization(manga_dict, user_list, global_std, global_average)
    

    manga_data = manga_rating_normalization_and_manga_weight(manga_dict, global_std, global_average)

    for manga in manga_data:
        manga_data[manga].extend(added_data[manga])#if I was to remake this whole loading this would not stay here, however computation is not my concern at this step
        
    return manga_dict, user_list, manga_data, manga_genres, genre_ids, global_std, global_average


#**********************************************************************
# Purpose: finding global average and standard deviation of ratings using all of the rating data
#
# Precondition: is passed a list of user rating data 
#               
# Postcondition: returns global standard deviation and global average 
#
#***********************************************************************
def global_normalization(user_list):
    giga_list = []

    for user in user_list:

        for manga in user:
            giga_list.append(user[manga][0])
    
    giga_list = np.array(giga_list)
    global_std = np.std(giga_list)
    global_average = np.average(giga_list)

    return  global_std, global_average


#**********************************************************************
# Purpose: give a user z-score to all manga ratings of users
#
# Note: The "user based z-score" is a z-score that uses all ratings one user has made to create a standard deviation and a average for that individual user. 
#       Using those two peices of information you can make a z-score made from the information of a user hence the name "user based z-score"
#
# Precondition: passed arguments for proper manga-first dictionary and user-first list. passed the global standard deviation and average 
#               
# Postcondition: creates a z-score based on the user's total rated manga if the user has rated less than 5 manga then the manga are rated using a z-score made using global normalization
#                each z-score is appended to the user-first list and appended to the manga first dictionary as well. every manga rating gets an individual calculation for it's user based z-score
#
#***********************************************************************
def users_rating_normalization(manga_dict, user_list, global_std, global_average):
    
    for ii, user in enumerate(user_list):

        rating_list = []
        for manga in user:
            rating_list.append(user[manga][0])

        rating_list = np.array(rating_list)
        std = np.std(rating_list)
        average = np.average(rating_list)
        
        for manga in user:
            
            if len(rating_list) > 5:

                min_std = 0.6

                z_score = (user[manga][0] - average) / max(std, min_std)
                z_score = float(z_score)
                user[manga].append(z_score)
                
                for rating in manga_dict[manga]:
                    if rating[0] == ii:
                        rating.append(z_score)

            else:
                z_score = ((user[manga][0] - global_average) / global_std)
                z_score = float(z_score)
                user[manga].append(z_score)
                
                for rating in manga_dict[manga]:
                    if rating[0] == ii:
                        rating.append(z_score)


#**********************************************************************
# Purpose: give a manga z-score to all manga ratings and returns a weight for each manga
#
# Note: The "manga based z-score" is a z-score that uses all ratings of a manga to create a standard deviation and a average for that individual manga. 
#       Using those two peices of information you can make a z-score made from the information of a lone manga hence the name "manga based z-score"
#
# 2nd Note: manga weight really should not be processed here. this function does two distinct actions because of it which makes the code less clear.
#           this is not an optimal way to code, however, it was an easy addition. If I was not the only person who will ever work on this code I would not leave it like this.
#
# Precondition: passed arguments for proper manga-first dictionary and passed the global standard deviation and average 
#               
# Postcondition: creates a z-score based on the total ratings of a manga if a manga has been rated less than 15 times then the manga ratings are rated using a z-score made using global normalization
#                each z-score is appended to the manga first dictionary. every rating gets an individual calculation for it's manga based z-score
#                The weight is a ratio which is larger the more ratings there are of the manga giving the manga more significance later in manga recomendations
#
#***********************************************************************
def manga_rating_normalization_and_manga_weight(manga_dict, global_std, global_average):
    manga_data = {}
    for manga in manga_dict:

        rating_list = []

        for user in manga_dict[manga]:
            rating_list.append(user[1])

        rating_list = np.array(rating_list)
        std = np.std(rating_list)
        average = np.average(rating_list)
        
        for user in manga_dict[manga]:
            
            if len(rating_list) >= 15:

                min_std = 0.6

                user.append(float((user[1]-average) / max(std, min_std)))
            else:
                user.append(float((user[1] - global_average) / global_std))
        

        weight = (len(rating_list)/(len(rating_list)+10))
        manga_data[manga] = [float(weight)]#[float(weight), float(std)]#std here was std.astype(float)

    return manga_data



``` 
Run the following Cell if you have no Username list    
Run the Cell after it instead if you already have run the first cell              and just want the same username list you got from your run of the cell directly bellow this one

This work how it does because a csv is made holding the username list to make it not necessary to request new users every time a username list is wanted to be used
```

In [ ]:
url = "https://myanimelist.net/users.php"

usernames = set()  # set ensures no duplicates
MAXUSERCOUNT = 500 #Actually an over estimate        ########## The number to change if you want more or less data! ##########  
                                                        # Does not actually guaranteed more more data, just more posibilities for more data

for _ in range(int(MAXUSERCOUNT/20)):
    # There is certainly a more efficent way of doing this however this is acceptable for my purposess
    try:
        response = requests.get(url, timeout=10)
       
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        for a in soup.select('a[href^="/profile/"]'): #searches the scraped page for hypertext refrence that is a profile
            usernames.add(a["href"].split("/profile/")[1]) # for each profile hypertext add it to the set (will elliminate duplicates)

    except Exception as ex:
        print(f"Error: {ex}")
    
    time.sleep(2)#wait interval
username_List = list(usernames)
print(f"Found {len(usernames)} users")

with open("usernames.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)

    for item in username_List:
        writer.writerow([item])

user_count = len(usernames)

In [5]:
with open("usernames.csv", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    username_List = list(reader)


print(f"Found {len(username_List)} users")


Found 35321 users


```
The following will only work correctly if username_list and or the usernames.csv match NameData.json.
This can be remedied by deleting NameData.json if you don't have a NameData.json there cannot be a problem, but requesting all of the data from myanimelist.com will take time depending on how many users you are requesting. 5,000 will likly take a couple of hours, 50,000 takes a full day 
```

In [6]:


manga_first_collection, user_first_collection, manga_data, manga_genres, genre_translation, global_std, global_average = make_collection(username_List)

datalist = {"manga_first_collection" : manga_first_collection, "user_first_collection" : user_first_collection, 
            "manga_data" : manga_data, "manga_genres" : manga_genres, "genre_translation" : genre_translation,
            "global_std": global_std, "global_average": global_average}

with open("datalist.json", "w") as file:
    json.dump(datalist, file)

In [1]:
import sys
print(sys.executable)

C:\Users\scott\AppData\Local\Programs\Python\Python312\python.exe


``` 
after successfuly running the cell above you can run the streamlit app via your terminal by first ensuring you are in your directory if 
type the following to get into a deeper folder: cd "your folder name" 
type the following to go up one folder: cd .. 
once you are in the correct directory type: streamlit run manga_recomendation.py
```

Bellow this is is un edited ramblings of mine which I used to figure out how I wanted to make my program


initialy I made my program use a bipartite graph, however, it did not work well because I did not save individual users which left a lot of important data that was very usefull in making the current product

In [ ]:
# I have code that gives me a list of users
# I have code that processes one user and give a matrix of shows with ratings
# need to create a graph whose edge weights are based on user reviews

# Problems to consider: people who only give 10, should probably be worth less because they are visably biased.
# If someone gives a show a 10 and another a 1 this should show that they are not similar types of shows, how do you do this? If it was negative their would be anti patterns.
# A user rates a show a 7 and another a 10 what is the weight that is used? Probably the average?
# What about a tie between a show that is 2 and another that is a 1? Both being bad does not indicate simalarity

# could:
# devalue a persons weight based on low spread of ratings, Would need to define low spread
# How many different types of ratings were made ie: (10,9,7,4)
# If someone does not have a wide range of ratings they should not be trusted as doing their best to rate shows
# With this method you also get people who only rate high and only rate low and not in between
# should also take into acount if a rating of 5 was only used once but they have rated 1000 manga
# 10 to a 10 is the best you can get (unless the user only gives 10's)
# should a 7 from a user who gives mostly 10s be qualified differently

# should probably do:
# Take the average of the users scores

# functional set up:
# using a list of names one by one process users and add their information to the network
# from one user there is manga and rating
# process all ratings user makes and calculate how they rank manga (look at "could:" section) Weight their rankings based on their gross ratings
# Node: count of each type of rating for that manga and overall rating which adds up (rating count * rating)for all ratings divided by count of all ratings (of the manga)
# Edge: use a swapable function to calculate what the edge score should be. Edge weight is calculated by total score divided by ratings

In [ ]:
# Scott Usher 
# Deciding what to do to get a good end product
# 8/21/2026
#
#   USE A CONST VARIABLE AS THE predicted COUNT OF USERS (the actuall count could be less than the const VAR but is not more)
#
# Desired functionality for a product-user
#     they enter hopefully 10 manga they like or dislike and get a list of manga returned that are from other users with similar likes and dislikes 
# The New List of functionality
# saving user specific data     to be able to find correlation between users 
#      Likely use a balanced binary search tree to find the selected manga in the users' lists.
#        If a user is found to have a sellected manga AND is similar enough to the rating of the product-user 
#          that users rating and name is put into a list with the rest of the results of that manga selection (or just a simple connection(pointer idk?) is put into the list)
#  after making at least one list of manga selection
#    Ranking of user importance: (Closeness to product-user )
#    *       Closeness to product-user.    Having as much overlap in oppinion to the product user as possible will presumably yeild suggestions which are similar to the product-user's implied taste 
#    *       Value of user ratings.        if a rated user is biased to rank manga with only 10 stars or 2 stars they do not have much imput on the range of quality of the shows they consume.
#    *       *                               The lower the variety of manga they have ranked the less their word means.         if they only have 8,5, and 1 star manga that means they have
#    *       *                                   good, mid, and bad as their only opinions making their information not as usefull as someone with more types of ratings.
#
# I want to scrape 10,000 users from my anime list     doing any processing on 10,000 users results in a lot of computation and a lot of memory use 
#
#     (I have realized here that I need a very effecient way of storing information if I wish to reasonably have expandable processing)
#     With my unreasonable perportions I think that I may not want to use a graph at all.
#
#     Numpy will be my savior
#     I can have a rating be 1 byte instead of 28 bytes (why do you do this python) 
#
# I believe that there will be more manga than users. and I will have to search for specific manga anyway no the other way arround.
#   Binary search tree of manga. from there use a reffrence of something else to tie a list of names to that manga                  Requires Python Sorted Containers or use c++ with actuall bst
#   *      finding manga will be Olog(n) and then just a run through a list to get all the rated users
#   *      this happens up to 10 times (probably allow for more). 
#   This is where things get dummy complex again to enable more evaluation methods to be used
#   * what do dynamic evaluations need to be used
#   *    Either need lists of loop-processed information as respective evaluation method's scores of their ratings. 
#   *    *    this change will make every single score be individualy tracked, which is simply needed for this method.
#   *    *    This can be done by using np.float16 in a 2d numpy array holding users and their eval method scores
#   *    OR need to process rated user's individual ratings as they are evaluated for closeness to the product-user 
#   *    *    This method would be likely a similar memory cost - eval method count* totalratings* -2 bytes which my estimate is 10000*500*10*2 bytes = 100MB less size without major redesigning 
#   *  a sorted container of manga with a np.array of users and their evaluations
#   *
#   From here the calculations of what is close can start
#   *   defo another passed in function to determine what calculation to do 
#   *   Make a numpy list to hold the count of ratings a user has in the selection.  only use the ones with relavence to the product-user in final processing
#   *   if all the numpy arrays use np.int16 to identify the users it can also be used as a way of ordering searches. (lets say) each new user is lastuser+1 for their identifier
#   *        the way they are added to lists makes them orderd from highest to lowest. thus doing a loop looking for the lowest name of that manga selection
#   *   i'm kinda done at this moment. could do a 10,000 long np.array of np.int16 with a row per processing type and a list of user numbers to search in
#            seems kinda dumb. could just do a normal list and continualy add a count of manga selections per user and tupples of the manga selection it's from and its score
# 
# 
# 
# 
#           
#

I have:
    data input
        node and edge creation
    
    poor edge weighting   
        Main thing that could use improvement: weight user rating usefullness 
            to keep data handling reasonable the information needed to rate users should be stored on graph creation to keep from refrencing users repeatedly.
            Cant keep each edge assigned to the user data must be cureved when the rating is given
            grab all of the ratings a user has made regradless of show (maybe in regard with later alorithims) and decide how to curve their scores based on their range of ratings.
    
    simple filtering



In [ ]:

# what is wanted as a product
#      enter in the name of a manga and get a result for manga that are rated similarly, whoses range can be windened or tightend by changing parameters of connection count and rating weight
#      The end product should have multiple different weights based on multiple different scoring methods so they can be swaped through quickly
# What is the goal from filtering?
#      The end product should be a graph of edges with many rating types with multiiple options for restrictions
#      From there the user should be able to limit what options are shown using the restriction and have the data and that recomended show shown to the user
# what is the difference between filtering the total options and just how they are displayed
#

#----------------------------------------------------------------------------------------------
# purpose: filtering and sorting graph 
#
#
#
def filtering(graph: nx.Graph, min_weight: float=0, title:str="Empty/False", lower_connection_count: int = 2, upper_connection_count = float("inf")) -> list:
    
    # filters out data
    filtered = (
        (a, b, data)
        for a,b, data in graph.edges(data=True)
           if title == "Empty/False" or a == title or b == title   # no title specified or if title is in the edge
               
                if data.get("SimpleW",0)>=min_weight      # larger than min weight
                
                    if data.get("count")>=lower_connection_count and data.get("count")<=upper_connection_count   # inbetween connection counts         
    )


    return heapq.nlargest(100, filtered, key=lambda edge: (edge[2]["count"], edge[2]["SimpleW"]))#this should probably be changed to not be hard coded and instead use input to decide the order
# edge is a 3 tuple (source_node, destination_node, attribute_dictionary) so edge[2] accesses the attribute_dictionary
# top would be a list of edge tuples 



